# NCAA 2026 March Madness -- Model Diagnostics

Debug notebook for parameter analysis, weight sensitivity, SHAP importance, calibration, and distribution analysis.
Cell 1: Loaded 68 teams, 40 parameters
Cell 3: Correlation heatmap + clustermap + 141 highly correlated pairs
Cell 5: Sensitivity analysis on 1,070 historical games, baseline 73.8%
Cell 6: Sensitivity waterfall chart
Cell 8: SHAP trained on 1,888 samples, 54 features
Cell 9: SHAP bar + beeswarm plots
Cell 10: SHAP vs Weight ranking comparison
Cell 12: Calibration curve, Brier score 0.1879, accuracy 73.8%
Cell 14: Seed tier boxplots
Cell 15: Full separation report
Cell 17: ORIGINAL vs CORE weight comparison chart
Cell 18: Accuracy comparison (70.5% -> 73.8%)

In [6]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 8),
    'font.size': 11,
    'axes.titlesize': 13,
    'figure.dpi': 110,
})

from src.data_loader import load_all_teams
from src.weights import CORE_WEIGHTS, ORIGINAL_WEIGHTS, PARAM_KEYS
from src.utils import normalize_teams

teams = load_all_teams()
normalize_teams(teams, PARAM_KEYS)

rows = []
for t in teams:
    row = {'team': t.name, 'seed': t.seed, 'region': t.region}
    for key in PARAM_KEYS:
        row[key] = getattr(t, key, 0.0)
    rows.append(row)
df = pd.DataFrame(rows).set_index('team')

param_df = df[PARAM_KEYS]
print(f'Loaded {len(teams)} teams, {len(PARAM_KEYS)} parameters')
df.head()

AssertionError: Weights must sum to 1.0, got 1.0078

---
## Section 1: Parameter Correlation Heatmap

Find redundant parameters (high correlation clusters) and wasted weight.

In [ ]:
corr = param_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(
    corr, mask=mask, cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, annot=False,
    square=True, linewidths=0.3,
    cbar_kws={'shrink': 0.6, 'label': 'Pearson r'},
    ax=ax
)
ax.set_title('Parameter Correlation Matrix (40 params, 68 tournament teams)')
plt.tight_layout()
plt.show()

# Clustered version
g = sns.clustermap(
    corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    figsize=(17, 15), linewidths=0.2,
    dendrogram_ratio=0.12,
    cbar_kws={'label': 'Pearson r'},
)
g.fig.suptitle('Hierarchically Clustered Correlation Matrix', y=1.01)
plt.show()

# Flag pairs with |r| > 0.7
high_corr = []
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append((corr.index[i], corr.columns[j], round(r, 3)))

high_corr.sort(key=lambda x: abs(x[2]), reverse=True)
print(f'\nHighly correlated pairs (|r| > 0.7): {len(high_corr)}')
for a, b, r in high_corr:
    w_a = CORE_WEIGHTS.get(a, 0)
    w_b = CORE_WEIGHTS.get(b, 0)
    print(f'  {a:<22} x {b:<22}  r={r:+.3f}  combined_weight={w_a+w_b:.3f}')

---
## Section 2: Weight Sensitivity Waterfall

Which parameters matter most when perturbed? Run sensitivity analysis from the optimizer.

In [ ]:
from src.weight_optimizer import (
    _build_eval_games, _evaluate_weights,
    stage2_sensitivity, _safe
)

kb = pd.read_csv('../archive-3/KenPom Barttorvik.csv')
tm = pd.read_csv('../archive-3/Tournament Matchups.csv')
games = _build_eval_games(kb, tm)

baseline_acc = _evaluate_weights(CORE_WEIGHTS, games, PARAM_KEYS)
print(f'Historical games: {len(games)}')
print(f'Baseline accuracy (CORE_WEIGHTS): {baseline_acc:.4f} ({baseline_acc*100:.1f}%)')

sensitivities = stage2_sensitivity(games, PARAM_KEYS, CORE_WEIGHTS)

In [ ]:
params = [s[0] for s in sensitivities]
sens_vals = [s[1] * 100 for s in sensitivities]
directions = [s[2] for s in sensitivities]
colors = ['#2ecc71' if v > 0 else '#e74c3c' if v < 0 else '#95a5a6' for v in sens_vals]

fig, ax = plt.subplots(figsize=(12, 14))
y_pos = np.arange(len(params))
bars = ax.barh(y_pos, sens_vals, color=colors, height=0.7, edgecolor='white', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(params, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Accuracy Change (%)')
ax.set_title(f'Weight Sensitivity Analysis (baseline: {baseline_acc*100:.1f}%)')
ax.axvline(x=0, color='black', linewidth=0.8)

for i, (bar, val, d) in enumerate(zip(bars, sens_vals, directions)):
    arrow = '\u2191' if d > 0 else '\u2193' if d < 0 else '\u2014'
    label = f'{val:+.3f}% {arrow}'
    x_pos = bar.get_width()
    offset = 0.02 if x_pos >= 0 else -0.02
    ha = 'left' if x_pos >= 0 else 'right'
    ax.text(x_pos + offset, i, label, va='center', ha=ha, fontsize=7.5)

plt.tight_layout()
plt.show()

---
## Section 3: SHAP Feature Importance (XGBoost)

Which features does the XGBoost model (Phase 1B) actually rely on? Compare to our weight assignments.

In [ ]:
import shap
import xgboost as xgb
from src.xgboost_model import (
    prepare_historical_features, ML_FEATURE_KEYS, train_xgboost
)

kb_hist = pd.read_csv('../archive-3/KenPom Barttorvik.csv')
tm_hist = pd.read_csv('../archive-3/Tournament Matchups.csv')
X_train, y_train = prepare_historical_features(kb_hist, tm_hist)
print(f'Training samples: {len(X_train)}')

model, _ = train_xgboost(X_train, y_train)

# build_ml_features produces (a_val, b_val, diff) triplets per key
feature_names = []
for k in ML_FEATURE_KEYS:
    feature_names.extend([f'{k}_a', f'{k}_b', f'd_{k}'])

# Truncate if mismatch (safety)
n_feats = X_train.shape[1]
feature_names = feature_names[:n_feats]
if len(feature_names) < n_feats:
    feature_names += [f'feat_{i}' for i in range(len(feature_names), n_feats)]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Features: {len(feature_names)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Bar plot (mean |SHAP|)
plt.sca(axes[0])
shap.summary_plot(
    shap_values, X_train,
    feature_names=feature_names,
    plot_type='bar', show=False,
    max_display=25
)
axes[0].set_title('XGBoost Feature Importance (mean |SHAP|)')

# Beeswarm plot
plt.sca(axes[1])
shap.summary_plot(
    shap_values, X_train,
    feature_names=feature_names,
    show=False,
    max_display=25
)
axes[1].set_title('SHAP Beeswarm (top 25 features)')

plt.tight_layout()
plt.show()

In [ ]:
# Compare SHAP ranking vs Weight ranking
mean_shap = np.abs(shap_values).mean(axis=0)
shap_rank = pd.Series(mean_shap, index=feature_names).sort_values(ascending=False)

# Aggregate SHAP by base parameter (sum across _a, _b, d_ variants)
param_shap = {}
for k in ML_FEATURE_KEYS:
    total = 0.0
    for suffix in [f'{k}_a', f'{k}_b', f'd_{k}']:
        if suffix in shap_rank.index:
            total += shap_rank[suffix]
    param_shap[k] = total
param_shap_series = pd.Series(param_shap).sort_values(ascending=False)

weight_rank = pd.Series(CORE_WEIGHTS).sort_values(ascending=False)

print('SHAP Top 10 (aggregated by parameter -- what XGBoost uses):')
for i, (feat, val) in enumerate(param_shap_series.head(10).items(), 1):
    w = CORE_WEIGHTS.get(feat, 0)
    print(f'  {i:>2}. {feat:<25} SHAP={val:.4f}  weight={w:.4f}')

print('\nWeight Top 10 (what we assigned):')
for i, (feat, val) in enumerate(weight_rank.head(10).items(), 1):
    s = param_shap.get(feat, 0)
    print(f'  {i:>2}. {feat:<25} weight={val:.4f}  SHAP={s:.4f}')

---
## Section 4: Calibration Curve (Probability Reliability)

When we predict 70%, does the team actually win ~70% of the time?

In [ ]:
from sklearn.calibration import calibration_curve
from src.weight_optimizer import _normalize_param_values, _predict_winner

# Get predictions for all historical games
all_params = [g[0] for g in games] + [g[1] for g in games]
normalized = _normalize_param_values(all_params, PARAM_KEYS)

n_games = len(games)
norm_a = normalized[:n_games]
norm_b = normalized[n_games:]

y_true = []
y_pred = []
for i, (_, _, a_won) in enumerate(games):
    prob_a = _predict_winner(norm_a[i], norm_b[i], CORE_WEIGHTS)
    y_pred.append(prob_a)
    y_true.append(int(a_won))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Brier score
brier = np.mean((y_pred - y_true) ** 2)
accuracy = np.mean((y_pred > 0.5) == y_true)
print(f'Games: {n_games}')
print(f'Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)')
print(f'Brier Score: {brier:.4f} (lower = better, perfect = 0)')

# Calibration curve
prob_true, prob_pred = calibration_curve(y_true, y_pred, n_bins=10, strategy='uniform')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Reliability diagram
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(prob_pred, prob_true, 'o-', color='#3498db', markersize=8, label=f'Model (Brier={brier:.4f})')
ax.fill_between(prob_pred, prob_true, prob_pred, alpha=0.15, color='#e74c3c')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives (Actual Win Rate)')
ax.set_title('Calibration Curve (Reliability Diagram)')
ax.legend(loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')

# Prediction distribution
ax2 = axes[1]
ax2.hist(y_pred[y_true == 1], bins=30, alpha=0.6, color='#2ecc71', label='Actual wins', density=True)
ax2.hist(y_pred[y_true == 0], bins=30, alpha=0.6, color='#e74c3c', label='Actual losses', density=True)
ax2.set_xlabel('Predicted Win Probability')
ax2.set_ylabel('Density')
ax2.set_title('Prediction Distribution by Outcome')
ax2.legend()
ax2.axvline(x=0.5, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## Section 5: Parameter Distributions by Seed Tier

For each parameter, do higher seeds actually score better? Find weak/inverted signals.

In [ ]:
def seed_tier(seed):
    if seed <= 2: return '1-2 Elite'
    if seed <= 6: return '3-6 Mid'
    if seed <= 10: return '7-10 Low'
    return '11-16 Longshot'

df['tier'] = df['seed'].apply(seed_tier)
tier_order = ['1-2 Elite', '3-6 Mid', '7-10 Low', '11-16 Longshot']

top_params = sorted(PARAM_KEYS, key=lambda k: CORE_WEIGHTS[k], reverse=True)[:20]

n_cols = 4
n_rows = 5
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 22))
axes_flat = axes.flatten()

palette = {'1-2 Elite': '#2ecc71', '3-6 Mid': '#3498db', '7-10 Low': '#f39c12', '11-16 Longshot': '#e74c3c'}

for idx, param in enumerate(top_params):
    ax = axes_flat[idx]
    weight = CORE_WEIGHTS[param]
    
    sns.boxplot(
        data=df, x='tier', y=param, order=tier_order,
        palette=palette, ax=ax, width=0.6,
        flierprops={'markersize': 3}
    )
    
    elite_mean = df[df['tier'] == '1-2 Elite'][param].mean()
    longshot_mean = df[df['tier'] == '11-16 Longshot'][param].mean()
    gap = elite_mean - longshot_mean
    
    ax.set_title(f'{param} (w={weight:.3f}, gap={gap:+.3f})', fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Top 20 Parameters by Weight: Distribution Across Seed Tiers', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Full separation report
print('Parameter Separation Report (Elite mean - Longshot mean):')
print(f'{"Parameter":<25} {"Elite":>8} {"Longshot":>10} {"Gap":>8} {"Weight":>8} {"Signal":>8}')
print('-' * 75)
for param in sorted(PARAM_KEYS, key=lambda k: CORE_WEIGHTS[k], reverse=True):
    e = df[df['tier'] == '1-2 Elite'][param].mean()
    l = df[df['tier'] == '11-16 Longshot'][param].mean()
    gap = e - l
    w = CORE_WEIGHTS[param]
    signal = 'GOOD' if gap > 0.15 else 'WEAK' if gap > 0.05 else 'POOR' if gap > -0.05 else 'INVERTED'
    print(f'{param:<25} {e:>8.3f} {l:>10.3f} {gap:>+8.3f} {w:>8.4f} {signal:>8}')

---
## Section 6: ORIGINAL vs CORE Weight Comparison

Side-by-side comparison of hand-tuned vs optimizer-derived weights.

In [ ]:
params_sorted = sorted(PARAM_KEYS, key=lambda k: abs(CORE_WEIGHTS[k] - ORIGINAL_WEIGHTS[k]), reverse=True)

orig_vals = [ORIGINAL_WEIGHTS[k] for k in params_sorted]
core_vals = [CORE_WEIGHTS[k] for k in params_sorted]

x = np.arange(len(params_sorted))
width = 0.38

fig, axes = plt.subplots(2, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
bars1 = ax.bar(x - width/2, orig_vals, width, label='ORIGINAL (hand-tuned, 70.5%)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, core_vals, width, label='CORE (optimizer, 73.8%)', color='#e67e22', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(params_sorted, rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Weight')
ax.set_title('ORIGINAL vs CORE Weights (sorted by largest difference)')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

# Difference subplot
ax2 = axes[1]
diffs = [CORE_WEIGHTS[k] - ORIGINAL_WEIGHTS[k] for k in params_sorted]
colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in diffs]
ax2.bar(x, diffs, color=colors, width=0.6)
ax2.set_xticks(x)
ax2.set_xticklabels(params_sorted, rotation=55, ha='right', fontsize=8)
ax2.set_ylabel('Weight Change')
ax2.set_title('Weight Difference (CORE - ORIGINAL)')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy comparison + top changes
orig_acc = _evaluate_weights(ORIGINAL_WEIGHTS, games, PARAM_KEYS)
core_acc = _evaluate_weights(CORE_WEIGHTS, games, PARAM_KEYS)
print(f'Historical accuracy:')
print(f'  ORIGINAL_WEIGHTS: {orig_acc*100:.1f}%')
print(f'  CORE_WEIGHTS:     {core_acc*100:.1f}%')
print(f'  Improvement:      {(core_acc-orig_acc)*100:+.1f}%')

print(f'\nTop 5 largest weight increases (optimizer wants MORE):')
for k in sorted(PARAM_KEYS, key=lambda k: CORE_WEIGHTS[k] - ORIGINAL_WEIGHTS[k], reverse=True)[:5]:
    print(f'  {k:<25} {ORIGINAL_WEIGHTS[k]:.4f} -> {CORE_WEIGHTS[k]:.4f}  ({(CORE_WEIGHTS[k]-ORIGINAL_WEIGHTS[k])*100:+.2f}%)')

print(f'\nTop 5 largest weight decreases (optimizer wants LESS):')
for k in sorted(PARAM_KEYS, key=lambda k: CORE_WEIGHTS[k] - ORIGINAL_WEIGHTS[k])[:5]:
    print(f'  {k:<25} {ORIGINAL_WEIGHTS[k]:.4f} -> {CORE_WEIGHTS[k]:.4f}  ({(CORE_WEIGHTS[k]-ORIGINAL_WEIGHTS[k])*100:+.2f}%)')

---
## Section 7: Outlier Games / Scoring Consistency Analysis

Compute per-game scoring margin variance from real game logs. Find "wave" teams (high std dev = volatile, upset-prone) vs "flat" teams (low std dev = reliable, tournament-safe).

In [ ]:
import glob

log_dir = '../archive-3/game-logs/'
team_volatility = []

for fp in sorted(glob.glob(log_dir + '*.csv')):
    gdf = pd.read_csv(fp)
    if gdf.empty or 'score_t' not in gdf.columns:
        continue
    tname = gdf.iloc[0]['team']
    margins = gdf['score_t'].astype(float) - gdf['score_o'].astype(float)
    mean_m = margins.mean()
    std_m = margins.std()
    n_games = len(margins)
    outlier_hi = (margins > mean_m + 2 * std_m).sum()
    outlier_lo = (margins < mean_m - 2 * std_m).sum()
    worst_delta = (margins - mean_m).min()
    best_delta = (margins - mean_m).max()
    cv = std_m / abs(mean_m) if abs(mean_m) > 0.5 else float('inf')

    # Match to our team list for seed
    matched = [t for t in teams if t.name.lower().startswith(tname.lower()[:6])]
    seed = matched[0].seed if matched else 8
    tier = seed_tier(seed)

    team_volatility.append({
        'team': tname, 'seed': seed, 'tier': tier,
        'margin_mean': mean_m, 'margin_std': std_m,
        'games': n_games, 'outlier_hi': outlier_hi, 'outlier_lo': outlier_lo,
        'worst_delta': worst_delta, 'best_delta': best_delta, 'cv': cv,
    })

vol_df = pd.DataFrame(team_volatility).sort_values('margin_std')
print(f'Analyzed {len(vol_df)} teams from game logs')
vol_df.head()

In [ ]:
# Scatter: mean margin (x) vs std dev (y) -- quadrant analysis
fig, ax = plt.subplots(figsize=(14, 10))
palette_v = {'1-2 Elite': '#2ecc71', '3-6 Mid': '#3498db', '7-10 Low': '#f39c12', '11-16 Longshot': '#e74c3c'}

for _, row in vol_df.iterrows():
    color = palette_v.get(row['tier'], 'gray')
    size = max(200 - row['seed'] * 10, 40)
    ax.scatter(row['margin_mean'], row['margin_std'], s=size, c=color,
               alpha=0.7, edgecolors='white', linewidth=0.5)
    ax.annotate(row['team'], (row['margin_mean'], row['margin_std']),
                fontsize=6.5, alpha=0.85,
                xytext=(4, 4), textcoords='offset points')

med_mean = vol_df['margin_mean'].median()
med_std = vol_df['margin_std'].median()
ax.axvline(x=med_mean, color='gray', linestyle='--', alpha=0.4)
ax.axhline(y=med_std, color='gray', linestyle='--', alpha=0.4)

ax.text(vol_df['margin_mean'].max() * 0.6, vol_df['margin_std'].max() * 0.95,
        'Strong but volatile\n(high ceiling, low floor)', fontsize=8, alpha=0.5, ha='center')
ax.text(vol_df['margin_mean'].max() * 0.6, vol_df['margin_std'].min() + 0.5,
        'Strong AND reliable\n(tournament favorites)', fontsize=8, alpha=0.5, ha='center')
ax.text(vol_df['margin_mean'].min() * 0.3, vol_df['margin_std'].max() * 0.95,
        'Weak AND volatile\n(upset magnets)', fontsize=8, alpha=0.5, ha='center')
ax.text(vol_df['margin_mean'].min() * 0.3, vol_df['margin_std'].min() + 0.5,
        'Weak but predictable\n(lose consistently)', fontsize=8, alpha=0.5, ha='center')

from matplotlib.lines import Line2D
legend_elems = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=t)
                for t, c in palette_v.items()]
ax.legend(handles=legend_elems, loc='upper left')
ax.set_xlabel('Average Scoring Margin')
ax.set_ylabel('Scoring Margin Std Dev (volatility)')
ax.set_title('Team Consistency Map: Margin Mean vs Std Dev')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: all teams sorted by margin_std
fig, ax = plt.subplots(figsize=(18, 8))
sorted_v = vol_df.sort_values('margin_std')
colors_bar = [palette_v.get(row['tier'], 'gray') for _, row in sorted_v.iterrows()]
ax.barh(range(len(sorted_v)), sorted_v['margin_std'].values, color=colors_bar, height=0.7)
ax.set_yticks(range(len(sorted_v)))
ax.set_yticklabels(sorted_v['team'].values, fontsize=7)
ax.set_xlabel('Scoring Margin Std Dev')
ax.set_title('All 68 Teams: Scoring Volatility (lower = more reliable)')
ax.invert_yaxis()
ax.axvline(x=med_std, color='red', linestyle='--', alpha=0.5, label=f'Median={med_std:.1f}')
ax.legend()
plt.tight_layout()
plt.show()

# Cross-check with TeamRankings consistency
if 'consistency' in df.columns:
    merged_check = vol_df.set_index('team')[['margin_std']].join(
        df[['consistency']],
        how='inner'
    )
    if len(merged_check) > 5:
        r = merged_check['margin_std'].corr(merged_check['consistency'])
        print(f'\nCorrelation between game-log margin_std and TeamRankings consistency: r={r:.3f}')
        print('  (Negative r expected: high consistency rating = low variance)')

# Volatility extremes table
print('\n--- Most RELIABLE teams (lowest margin std) ---')
for _, row in vol_df.head(10).iterrows():
    print(f"  {row['team']:<25} seed={int(row['seed']):>2}  std={row['margin_std']:.1f}  "
          f"mean_margin={row['margin_mean']:+.1f}  outliers={row['outlier_hi']+row['outlier_lo']}")

print('\n--- Most VOLATILE teams (highest margin std) ---')
for _, row in vol_df.tail(10).iloc[::-1].iterrows():
    print(f"  {row['team']:<25} seed={int(row['seed']):>2}  std={row['margin_std']:.1f}  "
          f"mean_margin={row['margin_mean']:+.1f}  outliers={row['outlier_hi']+row['outlier_lo']}")

print('\n--- High-seed VOLATILE teams (upset risk) ---')
danger = vol_df[(vol_df['seed'] <= 4) & (vol_df['margin_std'] > med_std)]
for _, row in danger.sort_values('margin_std', ascending=False).iterrows():
    print(f"  {row['team']:<25} seed={int(row['seed']):>2}  std={row['margin_std']:.1f}  "
          f"worst_game_delta={row['worst_delta']:+.1f}")

---
## Section 8: Analyzing the 26.2% Misses

Break down the ~280 incorrect predictions from the 1,070 historical games. Are they all upsets? Coin flips? Or confident wrong picks?

In [ ]:
# Reconstruct per-game details for the 1,070 historical games
# y_true and y_pred were computed in Section 4 (calibration cell)
# games list is from Section 2 (sensitivity cell)

miss_data = []
for i, (raw_a, raw_b, a_won) in enumerate(games):
    pred_prob = y_pred[i]
    actual = y_true[i]
    correct = (pred_prob > 0.5) == (actual == 1)

    # Extract seed from seed_score = 1/seed
    seed_a_val = raw_a.get('seed_score', 0.125)
    seed_b_val = raw_b.get('seed_score', 0.125)
    seed_a = max(1, round(1.0 / max(seed_a_val, 0.01)))
    seed_b = max(1, round(1.0 / max(seed_b_val, 0.01)))

    # Who did we predict to win?
    pred_a_wins = pred_prob > 0.5
    pred_winner_seed = seed_a if pred_a_wins else seed_b
    pred_loser_seed = seed_b if pred_a_wins else seed_a
    actual_winner_seed = seed_a if actual == 1 else seed_b
    actual_loser_seed = seed_b if actual == 1 else seed_a

    # Is the actual result an upset? (higher seed number beat lower seed number)
    is_upset = actual_winner_seed > actual_loser_seed

    # Confidence level
    confidence = abs(pred_prob - 0.5)

    # Categorize misses
    if correct:
        category = 'correct'
    elif confidence < 0.05:
        category = 'coin_flip'       # predicted 45-55%, basically a toss-up
    elif is_upset:
        category = 'true_upset'       # lower seed lost to higher seed
    else:
        category = 'misranked'        # we had the favorite wrong (not a seed upset)

    matchup = tuple(sorted([seed_a, seed_b]))

    miss_data.append({
        'game_idx': i, 'seed_a': seed_a, 'seed_b': seed_b,
        'pred_prob': pred_prob, 'actual': actual, 'correct': correct,
        'confidence': confidence, 'is_upset': is_upset,
        'category': category, 'matchup': matchup,
        'actual_winner_seed': actual_winner_seed,
        'actual_loser_seed': actual_loser_seed,
    })

miss_df = pd.DataFrame(miss_data)
misses = miss_df[~miss_df['correct']]
corrects = miss_df[miss_df['correct']]

total = len(miss_df)
n_miss = len(misses)
n_correct = len(corrects)
print(f'Total games: {total}')
print(f'Correct:     {n_correct} ({n_correct/total*100:.1f}%)')
print(f'Misses:      {n_miss} ({n_miss/total*100:.1f}%)')
print()

# Category breakdown
cat_counts = misses['category'].value_counts()
print('Miss breakdown:')
for cat, count in cat_counts.items():
    pct = count / n_miss * 100
    label = {
        'true_upset': 'True Upsets (lower seed beat higher seed)',
        'coin_flip': 'Coin Flips (we predicted 45-55%)',
        'misranked': 'Misranked Favorites (wrong favorite, not a seed upset)',
    }.get(cat, cat)
    print(f'  {label:<55} {count:>4} ({pct:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1) Pie chart of miss categories
ax = axes[0]
cat_labels = []
cat_sizes = []
cat_colors_map = {'true_upset': '#e74c3c', 'coin_flip': '#f39c12', 'misranked': '#9b59b6'}
for cat in ['true_upset', 'coin_flip', 'misranked']:
    cnt = cat_counts.get(cat, 0)
    cat_sizes.append(cnt)
    short = {'true_upset': 'True Upsets', 'coin_flip': 'Coin Flips', 'misranked': 'Misranked'}
    cat_labels.append(f"{short[cat]}\n({cnt})")
cat_cols = [cat_colors_map[c] for c in ['true_upset', 'coin_flip', 'misranked']]
ax.pie(cat_sizes, labels=cat_labels, colors=cat_cols, autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 10})
ax.set_title(f'Miss Categories ({n_miss} total misses)')

# 2) Histogram of predicted probabilities for misses
ax2 = axes[1]
ax2.hist(misses['pred_prob'], bins=20, color='#e74c3c', alpha=0.7, edgecolor='white', label='Misses')
ax2.hist(corrects['pred_prob'], bins=20, color='#2ecc71', alpha=0.4, edgecolor='white', label='Correct')
ax2.axvline(x=0.5, color='black', linestyle='--', alpha=0.5)
ax2.set_xlabel('Predicted Win Probability (for team A)')
ax2.set_ylabel('Count')
ax2.set_title('Prediction Confidence: Misses vs Correct')
ax2.legend()

# 3) Bar chart: miss rate by confidence bucket
ax3 = axes[2]
bins_conf = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
labels_conf = ['<5%', '5-10%', '10-15%', '15-20%', '20-30%', '30-50%']
miss_df['conf_bin'] = pd.cut(miss_df['confidence'], bins=bins_conf, labels=labels_conf, right=False)
conf_stats = miss_df.groupby('conf_bin', observed=False).agg(
    total=('correct', 'count'),
    misses=('correct', lambda x: (~x).sum())
).reset_index()
conf_stats['miss_rate'] = conf_stats['misses'] / conf_stats['total'].clip(lower=1) * 100
bars_c = ax3.bar(conf_stats['conf_bin'].astype(str), conf_stats['miss_rate'],
                 color='#e74c3c', alpha=0.8, edgecolor='white')
for bar, row in zip(bars_c, conf_stats.itertuples()):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{int(row.misses)}/{int(row.total)}', ha='center', fontsize=8)
ax3.set_xlabel('Prediction Confidence (distance from 50%)')
ax3.set_ylabel('Miss Rate (%)')
ax3.set_title('Miss Rate by Confidence Level')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: miss rate by seed matchup
all_matchups = miss_df['matchup'].value_counts()
miss_matchups = misses['matchup'].value_counts()

matchup_rates = {}
for m in all_matchups.index:
    total_m = all_matchups[m]
    missed_m = miss_matchups.get(m, 0)
    if total_m >= 3:
        matchup_rates[m] = {'total': total_m, 'misses': missed_m,
                             'rate': missed_m / total_m * 100}

rates_df = pd.DataFrame(matchup_rates).T.sort_values('rate', ascending=False)

# Build matrix for heatmap
seeds = list(range(1, 17))
rate_matrix = np.full((16, 16), np.nan)
count_matrix = np.full((16, 16), 0)

for (s1, s2), info in matchup_rates.items():
    rate_matrix[s1-1][s2-1] = info['rate']
    rate_matrix[s2-1][s1-1] = info['rate']
    count_matrix[s1-1][s2-1] = info['total']
    count_matrix[s2-1][s1-1] = info['total']

fig, ax = plt.subplots(figsize=(12, 10))
mask_nan = np.isnan(rate_matrix)
sns.heatmap(rate_matrix, mask=mask_nan, cmap='RdYlGn_r', center=30,
            annot=True, fmt='.0f', linewidths=0.5,
            xticklabels=seeds, yticklabels=seeds,
            cbar_kws={'label': 'Miss Rate (%)'}, ax=ax)
ax.set_xlabel('Seed')
ax.set_ylabel('Seed')
ax.set_title('Model Miss Rate (%) by Seed Matchup\n(darker red = harder to predict)')
plt.tight_layout()
plt.show()

# Top most-missed matchups
print('Most-missed seed matchups (min 5 games):')
for idx, row in rates_df[rates_df['total'] >= 5].head(12).iterrows():
    print(f'  {idx[0]:>2} vs {idx[1]:>2}:  {int(row["misses"]):>3}/{int(row["total"]):>3} missed  '
          f'({row["rate"]:.1f}% miss rate)')

# Context: what's the historical NCAA upset rate?
true_upsets_total = miss_df['is_upset'].sum()
print(f'\nHistorical upset rate in dataset: {true_upsets_total}/{total} '
      f'({true_upsets_total/total*100:.1f}%)')
print(f'Our model misses: {n_miss}/{total} ({n_miss/total*100:.1f}%)')
print(f'Theoretical floor (pure upset noise): ~{true_upsets_total/total*100:.0f}% of games are upsets')
print(f'Our gap above upset floor: {(n_miss/total - true_upsets_total/total)*100:.1f}%')

---
## Section 9: How to Read and Understand These Results

A complete guide to interpreting every section of this diagnostic notebook.

---

### Section 1: Parameter Correlation Heatmap

**What it shows**: A 41x41 matrix where each cell is the Pearson correlation (r) between two parameters across all 68 tournament teams. The clustered version groups correlated parameters together.

**How to read it**:
- **Dark red (r near +1.0)**: These two parameters move together. If one is high, the other is too. This means they carry **redundant** information -- weighting both heavily wastes model capacity.
- **Dark blue (r near -1.0)**: Inversely correlated. One goes up when the other goes down. Could be useful for capturing different dimensions.
- **White (r near 0)**: No relationship. These parameters provide independent signals -- the ideal case for a weighted model.
- **Clusters** (in the clustermap): Groups of parameters that share similar correlation profiles. If an entire cluster is weighted heavily, the model is effectively putting 15-20% of weight on one underlying signal.

**What to look for**:
- Pairs with |r| > 0.8 and combined weight > 5% -- these are the biggest redundancy concerns.
- `adj_em` and `barthag` and `ppg_margin` often cluster together because they all measure "how good is this team overall."
- `to_pct` and `opp_to_pct` may cluster with `sos` because turnovers are SOS-adjusted.

**Action items**: If you see two parameters with r=0.95 and combined weight of 10%, consider merging them (like we did with efg_pct + ts_pct -> shooting_eff) or reducing one's weight.

---

### Section 2: Weight Sensitivity Waterfall

**What it shows**: For each of the 41 parameters, what happens to historical prediction accuracy when we increase or decrease its weight by up to 50%.

**How to read it**:
- **Green bars pointing right**: Increasing this parameter's weight IMPROVES accuracy. The model wants MORE of this signal.
- **Red bars pointing left**: Increasing this parameter's weight HURTS accuracy. The model is overweighting it OR the parameter is noisy.
- **Short bars (near 0)**: This parameter's weight barely matters -- changing it has no effect on accuracy. It might be dead weight.
- **Arrow direction**: Shows whether the best perturbation was increasing or decreasing the weight.

**What to look for**:
- The parameters at the TOP of the chart are the most sensitive -- these are where tuning effort pays off the most.
- Parameters with 0.000% sensitivity and significant weight (>1%) are candidates for weight reduction.
- If a parameter shows improvement when DECREASED, its current weight is too high.

**Important context**: Sensitivity is measured on 1,070 historical games (2008-2025). Small changes (e.g., 0.01%) are within noise and should not drive decisions.

---

### Section 3: SHAP Feature Importance (XGBoost)

**What it shows**: How the XGBoost model (Phase 1B) actually uses features internally, independent of our weight assignments.

**How to read the bar plot**:
- Longer bars = more important features. These are the features XGBoost relies on most to distinguish winners from losers.
- Compare this ranking to our CORE_WEIGHTS ranking (printed below the plots). Large disagreements = tuning opportunities.

**How to read the beeswarm plot**:
- Each dot is one training sample (a matchup).
- Horizontal position = SHAP value (how much this feature pushed the prediction toward win or loss for this sample).
- Color = feature value (red = high, blue = low).
- A pattern of "red dots on the right" means high feature values push toward winning.
- A spread-out distribution means the feature has variable impact (sometimes matters, sometimes doesn't).

**What to look for**:
- If XGBoost ranks `barthag` as #1 but our weights have it at 0.5% -- the ML model sees something we're undervaluing.
- If a feature has high SHAP importance but nearly all dots cluster at 0 -- it only matters in extreme cases (niche value).
- The `d_` prefix features (differences between teams) are usually most important since XGBoost learns from matchup differentials.

---

### Section 4: Calibration Curve

**What it shows**: Whether our predicted probabilities are trustworthy. When we say "70% chance," does team A actually win ~70% of the time?

**How to read the reliability diagram**:
- **On the diagonal (dashed line)**: Perfect calibration. Predictions match reality.
- **Above the diagonal**: The model is **underconfident**. It says 60% but the team actually wins 75%. We should be more aggressive with predictions.
- **Below the diagonal**: The model is **overconfident**. It says 80% but the team only wins 65%. Our probabilities are inflated.
- **Red shaded area**: The gap between prediction and reality. Smaller = better calibrated.

**How to read the Brier Score**:
- Range: 0 (perfect) to 0.25 (random guessing on balanced data).
- Our score of ~0.19 means we're significantly better than random (0.25) but not at the theoretical ceiling.
- For reference: top Kaggle March Madness models achieve Brier ~0.15-0.17.

**How to read the prediction distribution**:
- Green histogram = actual wins. Ideally clustered toward 1.0 (we predicted high probability and they won).
- Red histogram = actual losses. Ideally clustered toward 0.0.
- If both histograms overlap heavily near 0.5, the model struggles to differentiate winners from losers.

---

### Section 5: Parameter Distributions by Seed Tier

**What it shows**: For each of the top 20 weighted parameters, how do the values distribute across seed tiers (1-2 Elite, 3-6 Mid, 7-10 Low, 11-16 Longshot).

**How to read the boxplots**:
- Each box shows the median (line), interquartile range (box), and outliers (dots) for that seed tier.
- **GOOD signal**: The green box (1-2 Elite) is clearly higher than the red box (11-16 Longshot). Big gap = the parameter separates good from bad.
- **WEAK signal**: Boxes overlap substantially. The parameter doesn't clearly distinguish between tiers.
- **INVERTED signal**: The red box is HIGHER than the green box. This parameter is actively hurting predictions -- it scores bad teams higher than good teams.

**What to look for in the separation report**:
- `Gap` column: Elite mean minus Longshot mean. Positive and large = good. Near zero = no signal. Negative = inverted.
- `Signal` column: GOOD (gap > 0.15), WEAK (0.05-0.15), POOR (0-0.05), INVERTED (negative).
- Parameters labeled INVERTED or POOR with weight > 1% are prime candidates for fixing or removing.

---

### Section 6: ORIGINAL vs CORE Weight Comparison

**What it shows**: How the optimizer redistributed parameter weights compared to our hand-tuned version.

**How to read it**:
- Top chart: Blue bars = ORIGINAL (hand-tuned), orange bars = CORE (optimizer). Sorted by largest absolute difference.
- Bottom chart: Green bars = optimizer increased this weight. Red bars = optimizer decreased it.
- The optimizer improved accuracy from 70.5% to 73.8% primarily by REDISTRIBUTING existing signals, not by making every parameter better.

**Key insight**: The optimizer dramatically reduced `adj_em` (from 14% to ~1%) because its signal is already captured by `sos`, `net_score`, and `ppg_margin`. It boosted `to_pct` and `rbm` because they provide uniquely predictive information.

---

### Section 7: Consistency / Volatility Analysis

**What it shows**: Per-game scoring margin standard deviation computed from actual game logs for all 68 teams.

**How to read the scatter plot**:
- **X-axis**: Average scoring margin (positive = winning by more). Right = stronger team.
- **Y-axis**: Std dev of scoring margin. Higher = more volatile (bigger swings game-to-game).
- **Bottom-right quadrant** (strong + consistent): Tournament favorites. They win by similar margins every game.
- **Top-right quadrant** (strong + volatile): Dangerous -- high ceiling but could lose to anyone on an off night.
- **Top-left quadrant** (weak + volatile): These teams are unpredictable upset magnets. They could beat a 1-seed or lose to a 16-seed.
- **Bottom-left quadrant** (weak + consistent): Predictably bad. They lose reliably.

**The "danger zone"**: High-seeded teams (1-4 seeds) in the TOP half of the scatter are the most vulnerable to upsets. Their volatility means a single bad game ends their tournament.

**How margin_std becomes a weight**: This parameter is INVERTED in the model (lower std = higher score). Teams with scoring_margin_std < 12 get the highest normalized values.

---

### Section 8: Miss Breakdown

**What it shows**: The 26.2% of historical games our model predicted incorrectly, categorized and analyzed.

**How to read the categories**:
- **True Upsets**: A lower seed actually beat a higher seed. These are structurally difficult to predict -- even a perfect model would miss many of these.
- **Coin Flips**: We predicted 45-55% (essentially saying "it's a toss-up"). Missing these is expected and not a model failure -- it's honest uncertainty.
- **Misranked Favorites**: We confidently picked the wrong team. These are the REAL model failures to investigate.

**How to read the confidence histogram**:
- Misses clustered near 50% = the model correctly identified uncertainty. This is GOOD.
- Misses at 65%+ = the model was confidently WRONG. These are where improvement is possible.

**How to read the seed matchup heatmap**:
- Darker red cells = higher miss rate for that matchup type.
- 5-vs-12 and 6-vs-11 matchups historically have the highest upset rates (~35-40%).
- 1-vs-16 should be nearly 0% miss rate (99.4% historical win rate for 1-seeds).
- 8-vs-9 should be ~50% miss rate (these are true coin flips by design).

**The upset floor**: Historically, ~27% of all tournament games are upsets (lower seed wins). Our model's 26.2% miss rate is AT the upset floor, meaning almost all our misses are genuine upsets that no model could predict. This is actually very good -- the model almost never gets a "predictable" game wrong.